# 🧬 SPS Self-Specialization — Minimal Research Prototype

This Colab demonstrates the core research idea in a simple, observable way:

**State 0 → self-replication → AI specialization → verification → State 1 → reuse**

The original capability is `IntegerMultiplication`. When the user requests a capability for `float × float` and none exists, the system creates a child, asks Ollama/Qwen to specialize it, verifies the generated code, and activates the result as `FloatMultiplication [S1]`.

## Research flow

```text
┌──────────────────────────────┐
│ IntegerMultiplication  [S0] │  Original static capability
└──────────────┬───────────────┘
               │ REPLICATE
               ▼
┌──────────────────────────────┐
│ IntegerMultiplication-child │  Runtime copy
│          [S0-C]              │
└──────────────┬───────────────┘
               │ SPECIALIZE with Ollama/Qwen
               ▼
┌──────────────────────────────┐
│   FloatMultiplication [S1]  │  Generated capability
└──────────────┬───────────────┘
               │ VERIFY ✓ + ACTIVATE ✓
               ▼
        Reuse on later requests
```

## 1. Clean environment and clone the latest `main`

We first remove stale copies so the experiment always uses the latest repository code.

In [ ]:
%cd /content
!pkill -9 ollama || true
!pkill -9 llama-server || true
!rm -rf /content/self-specialization
!git clone -q --branch main --single-branch https://github.com/muhammadnaumantahir/self-specialization.git
%cd /content/self-specialization
!echo 'Repository commit:'
!git rev-parse HEAD
!pip -q install -r requirements.txt pytest

## 2. Start Ollama safely

⚠️ Ollama is started from `/content`, **not** from the repository directory. This avoids the Colab `cannot get current path` failure when the repository is removed/recloned.

In [ ]:
%cd /content
!mkdir -p /content/ollama-work
!nohup ollama serve > /tmp/ollama.log 2>&1 &
!sleep 8
!echo 'Ollama:'
!ollama --version
!echo 'API status:'
!curl -sf http://127.0.0.1:11434/api/tags || (cat /tmp/ollama.log; exit 1)

## 3. Load the local coding model

The prototype uses `qwen2.5-coder:7b` locally through Ollama. No paid API is required.

In [ ]:
!ollama pull qwen2.5-coder:7b
!echo '
Available Ollama models:'
!ollama list

## 4. Run deterministic tests

These tests validate replication, specialization, verification, registration, and dispatch without depending on the live model.

In [ ]:
%cd /content/self-specialization
!PYTHONPATH=. pytest -q

## 5. Run the real self-specialization experiment

The output below is intentionally structured as a research demonstration. Pay special attention to **PHASE 3–5**, where the missing capability is detected, created, verified, activated, and reused.

In [ ]:
%cd /content/self-specialization
import os
os.environ['OLLAMA_MODEL'] = 'qwen2.5-coder:7b'
!PYTHONPATH=. python experiments/self_specialization_demo.py

## 🔎 How to read the result

### 1. `PHASE 1 — STATE 0`
`IntegerMultiplication [S0]` is the original statically programmed capability.

### 2. `PHASE 2`
An integer request is handled immediately by the existing S0 capability. No AI generation is needed.

### 3. `PHASE 3`
The float request reveals a missing contract: **`[float, float] → float`**. This is the trigger for self-specialization.

### 4. `PHASE 4`
Look for **NEW CAPABILITY CREATED AT RUNTIME**. This is the important evidence. It shows `FloatMultiplication`, its `S1` state, contract, parent ID, and the actual generated source code.

The **CAPABILITY LINEAGE** then shows the parent-child relationship, while **EVOLUTION EVENTS** show the sequence:

`REPLICATE → SPECIALIZE → GENERATED → VERIFY_PASS → ACTIVATE`

### 5. `PHASE 5`
A second float request resolves directly to `FloatMultiplication [S1]`. This demonstrates integration and reuse: Ollama is not called again.

## 🧪 What counts as the research evidence?

| Evidence | Meaning |
|---|---|
| `IntegerMultiplication [S0]` | Original capability |
| `IntegerMultiplication-child` | Runtime reproduction |
| `SPECIALIZE` | AI-driven transformation requested |
| `GENERATED: FloatMultiplication` | New capability produced |
| `VERIFY_PASS` | Generated capability passed tests |
| `FloatMultiplication [S1]` | New capability activated |
| Second float request | S1 is reused instead of regenerated |

**Important:** this minimal prototype stores the newly generated capability in the runtime registry. It does not yet persist it as a permanent `.py` file.

## 6. Ollama diagnostics — only if the experiment fails

If the real experiment fails, run the cell below and inspect the server process and log.

In [ ]:
%cd /content
!echo 'Current directory:'
!pwd
!echo '
Ollama processes:'
!ps aux | grep -E 'ollama|llama-server' | grep -v grep || true
!echo '
Ollama log:'
!cat /tmp/ollama.log

## Final expected conclusion

A successful run should end with:

```text
✓ State 0 capability existed before the float request
✓ State 0 reproduced itself as a child capability
✓ Ollama generated a specialized float capability
✓ Generated code passed verification
✓ FloatMultiplication was activated as State S1
✓ S1 was integrated into the registry
✓ A later float request reused S1 without AI generation

SUCCESS: State 0 reproduced, specialized into State 1, integrated, and reused.
```